# EMNIST: 영문자 + 숫자 인식

EMNIST ByMerge 데이터셋으로 47개 클래스 (0-9, A-Z, 일부 소문자) 인식 모델 학습

## 실행 전 확인사항
1. **런타임 → 런타임 유형 변경 → GPU (T4)** 선택
2. **런타임 → 모두 실행 (Ctrl+F9)** 으로 전체 실행

## 1. 환경 설정

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

# GPU 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. EMNIST 클래스 정의

In [ ]:
# ByMerge 47 클래스 라벨
EMNIST_LABELS = [
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9',  # 0-9: 숫자
    'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J',  # 10-19: 대문자
    'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T',  # 20-29
    'U', 'V', 'W', 'X', 'Y', 'Z',                      # 30-35
    'a', 'b', 'd', 'e', 'f', 'g', 'h', 'n', 'q', 'r', 't'  # 36-46: 일부 소문자
]

NUM_CLASSES = len(EMNIST_LABELS)
print(f"Classes: {NUM_CLASSES}")
print(f"Labels: {EMNIST_LABELS}")

## 3. 데이터 로드

In [ ]:
# EMNIST 이미지 전처리 (회전 + 반전 수정 필요)
class EMNISTTransform:
    def __init__(self):
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.1307,), (0.3081,))
        ])

    def __call__(self, img):
        # EMNIST는 이미지가 전치되어 있어서 회전 필요
        img = transforms.functional.rotate(img, -90)
        img = transforms.functional.hflip(img)
        return self.transform(img)

transform = EMNISTTransform()

# 데이터셋 다운로드
train_dataset = datasets.EMNIST(
    './data',
    split='bymerge',
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.EMNIST(
    './data',
    split='bymerge',
    train=False,
    download=True,
    transform=transform
)

# DataLoader
BATCH_SIZE = 128
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

print(f"Train data: {len(train_dataset):,}")
print(f"Test data: {len(test_dataset):,}")

## 4. 샘플 데이터 확인

In [ ]:
# 샘플 이미지 표시
fig, axes = plt.subplots(2, 10, figsize=(15, 4))

for i in range(20):
    idx = np.random.randint(len(train_dataset))
    img, label = train_dataset[idx]

    ax = axes[i // 10, i % 10]
    ax.imshow(img.squeeze(), cmap='gray')
    ax.set_title(f"'{EMNIST_LABELS[label]}'", fontsize=10)
    ax.axis('off')

plt.suptitle('EMNIST Sample Images', fontsize=14)
plt.tight_layout()
plt.show()

## 5. 모델 정의

In [ ]:
class CNN(nn.Module):
    """
    EMNIST용 CNN 모델 (47 클래스)
    3개의 Conv 레이어 + BatchNorm + Dropout
    """
    def __init__(self, num_classes=47):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(128 * 3 * 3, 256)
        self.fc2 = nn.Linear(256, num_classes)
        self.dropout = nn.Dropout(0.3)
        self.bn1 = nn.BatchNorm2d(32)
        self.bn2 = nn.BatchNorm2d(64)
        self.bn3 = nn.BatchNorm2d(128)

    def forward(self, x):
        x = self.pool(F.relu(self.bn1(self.conv1(x))))  # (B, 32, 14, 14)
        x = self.pool(F.relu(self.bn2(self.conv2(x))))  # (B, 64, 7, 7)
        x = self.pool(F.relu(self.bn3(self.conv3(x))))  # (B, 128, 3, 3)
        x = x.view(-1, 128 * 3 * 3)
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

model = CNN(num_classes=NUM_CLASSES).to(device)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 6. 학습 설정

In [ ]:
EPOCHS = 15
LEARNING_RATE = 0.001

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

print(f"Epochs: {EPOCHS}")
print(f"Learning rate: {LEARNING_RATE} (halved every 5 epochs)")
print(f"Optimizer: Adam")
print(f"Loss: CrossEntropyLoss")

## 7. 학습 실행

In [ ]:
def train_epoch(model, device, train_loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        pred = output.argmax(dim=1)
        correct += pred.eq(target).sum().item()
        total += target.size(0)

    return total_loss / len(train_loader), 100. * correct / total


def evaluate(model, device, test_loader, criterion):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            total_loss += criterion(output, target).item()
            pred = output.argmax(dim=1)
            correct += pred.eq(target).sum().item()
            total += target.size(0)

    return total_loss / len(test_loader), 100. * correct / total


# 학습 실행
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}
best_acc = 0

print("="*60)
print("Training Start")
print("="*60)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_epoch(model, device, train_loader, optimizer, criterion)
    test_loss, test_acc = evaluate(model, device, test_loader, criterion)
    scheduler.step()

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['test_loss'].append(test_loss)
    history['test_acc'].append(test_acc)

    best_marker = "*" if test_acc > best_acc else ""
    if test_acc > best_acc:
        best_acc = test_acc

    print(f"Epoch {epoch:2d}/{EPOCHS} | "
          f"Train: {train_acc:.2f}% | "
          f"Test: {test_acc:.2f}% {best_marker}")

print("="*60)
print(f"Training Complete! Best accuracy: {best_acc:.2f}%")
print("="*60)

## 8. 결과 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], 'b-', label='Train', linewidth=2)
axes[0].plot(history['test_loss'], 'r-', label='Test', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('EMNIST Loss Curve')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], 'b-', label='Train', linewidth=2)
axes[1].plot(history['test_acc'], 'r-', label='Test', linewidth=2)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_title('EMNIST Accuracy Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('emnist_learning_curves.png', dpi=150)
plt.show()

## 9. 예측 테스트

In [ ]:
# 랜덤 샘플 예측
model.eval()
fig, axes = plt.subplots(3, 10, figsize=(15, 5))

with torch.no_grad():
    for i in range(30):
        idx = np.random.randint(len(test_dataset))
        img, label = test_dataset[idx]

        output = model(img.unsqueeze(0).to(device))
        pred = output.argmax(dim=1).item()
        conf = F.softmax(output, dim=1)[0, pred].item()

        ax = axes[i // 10, i % 10]
        ax.imshow(img.squeeze(), cmap='gray')

        color = 'green' if pred == label else 'red'
        ax.set_title(f"'{EMNIST_LABELS[pred]}'\n{conf:.0%}", fontsize=9, color=color)
        ax.axis('off')

plt.suptitle('Prediction Results (Green=Correct, Red=Wrong)', fontsize=14)
plt.tight_layout()
plt.show()

## 10. 모델 저장

In [ ]:
# 모델 저장
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'history': history,
    'final_accuracy': history['test_acc'][-1],
    'best_accuracy': best_acc,
    'num_classes': NUM_CLASSES,
    'labels': EMNIST_LABELS
}, 'emnist_cnn.pt')

print("Model saved: emnist_cnn.pt")

# 가중치만 저장
torch.save(model.state_dict(), 'emnist_weights.pth')
print("Weights saved: emnist_weights.pth")

## 11. Google Drive 저장 (선택)

In [ ]:
# Google Drive 마운트
from google.colab import drive
drive.mount('/content/drive')

# 저장 경로
import shutil
save_dir = '/content/drive/MyDrive/AI_Practice/02-EMNIST/models'
import os
os.makedirs(save_dir, exist_ok=True)

# 파일 복사
shutil.copy('emnist_cnn.pt', f'{save_dir}/emnist_cnn.pt')
shutil.copy('emnist_weights.pth', f'{save_dir}/emnist_weights.pth')
shutil.copy('emnist_learning_curves.png', f'{save_dir}/../results/emnist_learning_curves.png')

print(f"Files saved to: {save_dir}")

## 12. 다운로드 (PC로)

In [ ]:
from google.colab import files

# 모델 다운로드
files.download('emnist_cnn.pt')
files.download('emnist_learning_curves.png')

---

## 완료!

다운로드한 `emnist_cnn.pt` 파일을 로컬 PC의 `02-EMNIST/models/` 폴더에 넣고:

```bash
cd experiments/exp00_baseline
python font_test.py  # PC 폰트 테스트
python test.py       # 손글씨 테스트
```